# Industrial-VLM: QLoRA Fine-Tuning Pipeline
**Model**: LLaVA-1.5-7B | **Method**: QLoRA (4-bit NF4) | **Dataset**: MVTec AD (15 categories)

This notebook runs the complete pipeline:
1. Data preprocessing (RGB conversion, 336×336 resize, stratified split)
2. Zero-shot baseline evaluation
3. Weights & Biases Authentication
4. QLoRA fine-tuning (r=32, α=64, Q-K-V-O projections)
5. Fine-tuned evaluation
6. Auto-push results to GitHub
7. Artifact backup to WandB

**Requirements**: Kaggle GPU T4 x2, MVTec AD dataset attached.

## Cell 1: Setup & Environment Initialization

In [ ]:
!git clone https://github.com/dvydinh/vlm-industrial-finetuner.git
!pip install -q -r vlm-industrial-finetuner/requirements.txt
!pip install wandb -U

## Cell 2: Data Preprocessing (ETL Pipeline)
Converts MVTec AD from unsupervised format to supervised instruction-tuning JSONL.
- Grayscale -> RGB conversion
- 1024x1024 -> 336x336 resize
- Stratified 80/20 train/test split
- Context-anchored prompts with item category tagging.

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/data_builder.py \
    --data_dir /kaggle/input/datasets/ipythonx/mvtec-ad \
    --output_dir /kaggle/working/processed_data

## Cell 3: Zero-Shot Baseline Evaluation
Evaluates the base LLaVA-1.5-7B (without LoRA adapters) to establish a performance baseline.
Metrics are exported to `/kaggle/working/results/eval_baseline.json`.

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/evaluate.py \
    --baseline \
    --test_data /kaggle/working/processed_data

## Cell 4: Weights & Biases Authentication
Provides session credentials for remote logging and artifact storage.

In [ ]:
import wandb

# Initialize W&B API Key configuration
wandb.login(key="YOUR_WANDB_API_KEY_HERE")

## Cell 5: QLoRA Fine-Tuning Execution
Executes PEFT configuration targeting Q-K-V-O self-attention modules.
Training logs are periodically exported to `/kaggle/working/results/training_log.json`.

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/train.py \
    --dataset /kaggle/working/processed_data \
    --output_dir /kaggle/working/lora_weights \
    --epochs 2 \
    --lr 2e-5

## Cell 6: Fine-Tuned Model Evaluation
Evaluates the combined Base Model + newly trained LoRA adapters against the test partition.
Metrics are exported to `/kaggle/working/results/eval_finetuned.json`.

In [ ]:
!python /kaggle/working/vlm-industrial-finetuner/src/evaluate.py \
    --model_dir /kaggle/working/lora_weights \
    --test_data /kaggle/working/processed_data

## Cell 7: Repository Metric Synchronization
Uploads the generated evaluation data points (`results/*.json`, `results/*.csv`) directly to the project's main branch.
Note: A personal access token with `repo` scope is required.

In [ ]:
import subprocess, os

# Repository Configuration
GITHUB_TOKEN = "ghp_YOUR_TOKEN_HERE"

# Initialize global git configuration
subprocess.run(["git", "config", "--global", "user.email", "dvydinh@users.noreply.github.com"], check=True)
subprocess.run(["git", "config", "--global", "user.name", "dvydinh"], check=True)

# Stage evaluation artifacts
os.makedirs("vlm-industrial-finetuner/results", exist_ok=True)
subprocess.run("cp /kaggle/working/results/*.json vlm-industrial-finetuner/results/", shell=True)
subprocess.run("cp /kaggle/working/results/*.csv vlm-industrial-finetuner/results/", shell=True)

# Execute downstream git synchronization
os.chdir("vlm-industrial-finetuner")
subprocess.run(["git", "add", "results/"], check=True)
subprocess.run(["git", "commit", "-m", "results: auto-upload evaluation metrics from Kaggle"], check=True)
subprocess.run(["git", "push", f"https://{GITHUB_TOKEN}@github.com/dvydinh/vlm-industrial-finetuner.git", "main"], check=True)
print("[SYSTEM] Evaluation files synchronized to GitHub successfully.")

## Cell 8: WandB Artifact Archival
Generates a consolidated artifact of the Kaggle pipeline output (both weights and numerical logs) for long-term storage.

In [ ]:
import wandb

print("[SYSTEM] Initializing persistent workspace archival to WandB...")
run = wandb.init(project="vlm-industrial-finetuner", name="session_archival")

artifact = wandb.Artifact(name="full-kaggle-workspace", type="backup")
artifact.add_dir("/kaggle/working/lora_weights", name="lora_weights")
artifact.add_dir("/kaggle/working/results", name="results_and_logs")

run.log_artifact(artifact)
run.finish()
print("[SYSTEM] Archival process completed successfully.")